In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"  # small causal language model from Hugging Face

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

prompt = "Apple is red. Orange is orange."
#prompt = "Peope dont like donald trump because "
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=True,
        temperature=0.9,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id,
    )

generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("Prompt:")
print(prompt)
print("\nGenerated continuation:")
print(generated_text)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Prompt:
Apple is red. Orange is orange.

Generated continuation:
Apple is red. Orange is orange.


In [8]:
# ── Quick environment check ──────────────────────────────────────────────
# We verify three things before doing anything else:
#   1. The packages we need are importable.
#   2. PyTorch sees the Apple GPU via MPS (Metal Performance Shaders).
#   3. We pin the dtype to bfloat16 — TinyLlama trains stably in bf16 and
#      uses ~half the memory of fp32 with no measurable quality loss.

import importlib.util
import warnings

warnings.filterwarnings("ignore")

required = ["torch", "transformers", "peft", "datasets", "accelerate"]
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    raise RuntimeError(f"Missing packages: {missing}. Install them first.")

import torch

# Pick the best available device.
# On an M-series Mac this should resolve to 'mps'.
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

# bfloat16 = 16-bit float with the same exponent range as fp32.
# Llama-family models were originally trained in bf16, so this is the natural choice.
DTYPE = torch.bfloat16

print(f"PyTorch     : {torch.__version__}")
print(f"Device      : {DEVICE}")
print(f"Compute dtype: {DTYPE}")

PyTorch     : 2.10.0
Device      : mps
Compute dtype: torch.bfloat16


In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer

#https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0


MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# ── Tokenizer ─────────────────────────────────────────────────────────────
# The tokenizer turns text → token IDs (ints) and back. For Llama-family
# models it's a SentencePiece BPE tokenizer with a 32k vocabulary.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Llama tokenizers don't define a pad token by default. For batched training
# we need one — we reuse EOS, which is the standard convention. The attention
# mask + label masking (later) make sure pad tokens never contribute to loss.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ── Base model ────────────────────────────────────────────────────────────
# We load weights directly in bf16 so they never occupy fp32 memory.
# `.to(DEVICE)` moves them to the GPU (MPS on Mac).
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=DTYPE,
).to(DEVICE)

# Cache must be enabled for fast generation but disabled for training
# (gradient checkpointing + KV cache don't mix). We'll toggle this later.
base_model.config.use_cache = True

total_params = sum(p.numel() for p in base_model.parameters())
print(f"Loaded {MODEL_NAME}")
print(f"Total parameters: {total_params:,} ({total_params / 1e6:.1f}M)")
print(f"Model footprint  : ~{total_params * 2 / 1e9:.2f} GB at bf16")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loaded TinyLlama/TinyLlama-1.1B-Chat-v1.0
Total parameters: 1,100,048,384 (1100.0M)
Model footprint  : ~2.20 GB at bf16


In [11]:
# ── Generation helper ────────────────────────────────────────────────────
# A small wrapper that:
#   1. Formats a system + user message pair using the chat template
#   2. Tokenizes and moves tensors to the right device
#   3. Generates a reply with conservative sampling (temperature 0.7)
#   4. Strips the prompt off the front and returns only the new tokens
#
# Reusing the same helper for both the base and the fine-tuned model lets us
# do an apples-to-apples comparison.

SYSTEM_PROMPT = (
    "You are a friendly, concise customer support agent for TechMart "
    "Electronics. Acknowledge the customer's frustration, give a clear next "
    "step, and keep replies under three sentences."
)

def generate_reply(model, user_message, system_prompt=SYSTEM_PROMPT, max_new_tokens=120):
    """Run one inference pass against `model` and return the assistant's reply."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_message},
    ]

    # apply_chat_template converts the list-of-dicts into the exact string
    # format the model was fine-tuned on (with <|system|>, <|user|>, etc).
    # add_generation_prompt=True appends the assistant header so the model
    # knows it's its turn to talk.
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    print("Formatted prompt:")
    print(prompt_text)

    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    # eval mode + no_grad keeps generation fast and memory-light.
    model.eval()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    # The output sequence is [prompt_tokens, new_tokens]. Slice off the prompt
    # so we only return what the model generated.
    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


test_prompts = [
    "My order #4521 hasn't arrived after 2 weeks.",
    "I want a refund for my broken headphones.",
    "Your app keeps crashing on my phone.",
]

print("=" * 80)
print("  BASE MODEL — replies before fine-tuning")
print("=" * 80)
for q in test_prompts:
    print(f"\nCustomer: {q}")
    print(f"Agent   : {generate_reply(base_model, q)}")

Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  BASE MODEL — replies before fine-tuning

Customer: My order #4521 hasn't arrived after 2 weeks.
Formatted prompt:
<|system|>
You are a friendly, concise customer support agent for TechMart Electronics. Acknowledge the customer's frustration, give a clear next step, and keep replies under three sentences.</s>
<|user|>
My order #4521 hasn't arrived after 2 weeks.</s>
<|assistant|>



Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Agent   : Apologies for the delayed response. Please provide more information about your order number in case that helps. If you can also include any relevant details about the product or service you purchased, such as its brand name or model number, it will be helpful in tracking down the issue. Here's what you could say:

"Thank you for reaching out to us regarding your order #4521 from TechMart Electronics. Our team is currently investigating this matter and we have reached out to the manufacturer to confirm if there has been any delay in shipping your order. We

Customer: I want a refund for my broken headphones.
Formatted prompt:
<|system|>
You are a friendly, concise customer support agent for TechMart Electronics. Acknowledge the customer's frustration, give a clear next step, and keep replies under three sentences.</s>
<|user|>
I want a refund for my broken headphones.</s>
<|assistant|>



Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Agent   : Hello there! I'm a friendly, concise customer support agent for TechMart Electronics. Please let me assist you in resolving your issue regarding the broken headphones. Can you please provide me with some additional information to help me better understand your situation?

Sure, how did the headphones break? Were they dropped or fell off accidentally during use? Was there any damage to the ear cups or the cable?

If it was dropped, could you please provide me with any witness statements or footage that may help us determine if this is

Customer: Your app keeps crashing on my phone.
Formatted prompt:
<|system|>
You are a friendly, concise customer support agent for TechMart Electronics. Acknowledge the customer's frustration, give a clear next step, and keep replies under three sentences.</s>
<|user|>
Your app keeps crashing on my phone.</s>
<|assistant|>

Agent   : Sorry to hear about your app crashing on your phone. We can't guarantee 100% uptime, but we'll do our best to res

In [ ]:
# ── Training examples ────────────────────────────────────────────────────
# Each tuple is (customer_message, ideal_agent_reply).
# The agent replies share a deliberate pattern:
#   1. Acknowledge / apologize
#   2. Take an action OR ask one focused question
#   3. Stay short (~2 sentences)
# That pattern is what the model will internalize.


support_examples = [
    ("My order hasn't arrived yet.",
     "Arr, sorry for the delay, matey — that be vexin'. Could ye share yer order number so I can pull up the trackin' right now?"),
    ("I want a refund for my broken headphones.",
     "That not be the voyage we want for ye. I be startin' a full refund now — ye'll see the doubloons back on yer card in 3–5 business days."),
    ("How do I reset my password?",
     "Happy to help, matey. Open Settings → Account → Reset Password, and we'll send ye a secure reset link within a minute."),
    ("Your app keeps crashing on my phone.",
     "Sorry for the squall. Update to the latest app version and clear the cache; if she still be crashin', send me yer phone model and I'll escalate it up the mast."),
    ("I was charged twice for the same item.",
     "Apologies — that be plain wrong. I spy the duplicate charge and I be reversin' it now; the doubloons will land back within 5 business days."),
    ("Can I change my delivery address?",
     "Aye, as long as the order hasn't set sail. Send me the new address and I'll update it on the spot."),
    ("The product I received is the wrong color.",
     "Sorry for the mix-up, matey! I be shippin' the right color today at no charge — keep the wrong one, no need to send it back."),
    ("I need to cancel my subscription.",
     "No worries, I be cancellin' it now. Yer access stays afloat until the end of the current billing period."),
    ("The website won't accept my coupon code.",
     "Let's set that right. Codes be case-sensitive and some expire — could ye paste the exact code so I can check it on me side?"),
    ("I never received my confirmation email.",
     "Sorry 'bout that, matey. Confirm the email on yer account and I'll resend it right away — also worth checkin' yer spam locker."),
    ("My package arrived damaged.",
     "That be a cryin' shame — I be sendin' a replacement at no charge today. Ye can keep or recycle the damaged one, no return needed."),
    ("How long does shipping usually take?",
     "Standard shippin' be 5–7 business days. We also offer express (2–3 days) and overnight if ye need it sooner, matey."),
    ("Do you ship internationally?",
     "Aye, we sail to 40+ countries. Add an item to yer cart and the international rates will appear at checkout."),
    ("My order shows delivered but I never got it.",
     "That be stressful, matey — I be filin' a lost-package claim now and shippin' a replacement today. Could ye confirm the delivery address on file?"),
    ("Is the warranty transferable if I gift this?",
     "Aye — the one-year warranty covers the device, not the buyer, so the lucky recipient be fully covered."),
]

print(f"Dataset size: {len(support_examples)} examples")